# Apache Arrow Flight

![Arrow Logo](images/arrow.png)

Arrow is a foundational technology in the Data Engineering space. It powers all your favourite tools
 from Spark to Polars, Duckdb and Snowflake. As they say, it's a standard

![XKDC Standards](images/Standards.png)

Why this particular standard matters, is that it solves the problem of interprocess communication
 between different languages and frameworks.

## Process interop

Take the scenario of the Spark UDF in the old days:

![ipc](images/spark_udf.png)

Without a standard for memory layout for the data, we need to introduce glue code between the JVM
Spark Memory and the Pandas Numpy memory layout, usually having to copy the data back and forth.


When we introduce Apache Arrow, both runtimes can share the same memory layout, and we can avoid
the need for any glue code or copying.
![ipc](images/spark_arrow.png)


This is true for any library which uses Arrow, like Duckdb, Pandas and Polars.

![I made this](images/i_made_this.jpg)


## Arrow as the data interchange format
Having Arrow as an universal data interchange format allows libraries to delegate responsibility for their memory layout and compute to 
Arrow and instead focus on their value-adding layer. 

We've seen this before - compilers came along, and gave us all these different optimizations and now we no longer inline statements or unravel loops. 

LLVM and JVM are both examples of the power of separating the layers of a program, so that we can focus on the value-adding part.

So Arrow is great - but what is Arrow Flight then?

# Why Flight? - in action

Let's compare to a normal REST service, which might be more familiar to more of you.

In both scenarios, we have 10,000,000 rows we want to be able to fetch, and we
want to convert the result to a dataframe.

## REST
A standard FastAPI-based REST API that you've probably written hundreds of.

Guesses? Problems?

In [1]:
import httpx2
import polars as pl

In [7]:
%%timeit -r 2
req = httpx2.get("http://rest:8000/data/rides/all", headers={"Authorization": "Bearer pydata_amsterdam"})
pl.from_records(req.json())

ReadTimeout: timed out

## Why do we care about Arrow Flight?

In [14]:
from flight_server.server import Server  # noqa
from pyarrow import flight

In [15]:
client = flight.connect("grpc+tls://arrow-flight-server.fly.dev:443")

In [16]:
info = client.get_flight_info(flight.FlightDescriptor.for_path("rides"))

FlightServerError: forbidden: User: arn:aws:iam::449650107887:user/arrow_flight_backend is not authorized to perform: s3tables:GetTableMetadataLocation on resource: arn:aws:s3tables:eu-north-1:449650107887:bucket/anders-pydata-demo/table/* because no identity-based policy allows the s3tables:GetTableMetadataLocation action. Detail: Failed

In [3]:
client = flight.connect("grpc://server:7000")
info = client.get_flight_info(flight.FlightDescriptor.for_path("rides"))

In [4]:
%%timeit -r 1
reader = client.do_get(info.endpoints[0].ticket)
pl.from_arrow(reader.read_all())

306 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


## Apache Arrow, but for servers

In essence, Arrow Flight tries to solve the same issue as Arrow, but at the Client/Server level instead of
just between processes on the same machine. We have the same problem of needing to copy the data
between DB wire protocol, ODBC and Pandas Numpy.

![DB ODBC](images/db_odbc.png)


Or in the REST example, backend format → JSON → Pandas.

![JSON API](images/api_json.png)

To make it even worse, both of these data representations are row-based, and we are targeting a column-based data format.

![Table Data](images/table_data.png)

# But wait, there's more! We also get free stuff!

Overall, a little bit more boilerplate if you're not used to Arrow Flight, but let's look at
what extras we got in the exchange, apart from the speedup.

Firstly, Arrow Flight provides great discoverability and metadata before we commit to loading data.

Let's have a closer look at what the `info` object can give us

In [4]:
info.schema

ride_id: large_string
  -- field metadata --
  PARQUET:field_id: '1'
rideable_type: large_string
  -- field metadata --
  PARQUET:field_id: '2'
started_at: timestamp[us]
  -- field metadata --
  PARQUET:field_id: '3'
ended_at: timestamp[us]
  -- field metadata --
  PARQUET:field_id: '4'
start_station_name: large_string
  -- field metadata --
  PARQUET:field_id: '5'
start_station_id: large_string
  -- field metadata --
  PARQUET:field_id: '6'
end_station_name: large_string
  -- field metadata --
  PARQUET:field_id: '7'
end_station_id: large_string
  -- field metadata --
  PARQUET:field_id: '8'
start_lat: double
  -- field metadata --
  PARQUET:field_id: '9'
start_lng: double
  -- field metadata --
  PARQUET:field_id: '10'
end_lat: double
  -- field metadata --
  PARQUET:field_id: '11'
end_lng: double
  -- field metadata --
  PARQUET:field_id: '12'
member_casual: large_string
  -- field metadata --
  PARQUET:field_id: '13'

The schema of the data is readily available. Did we get the correct dataset?
Does it have what I need?

In [5]:
print(
    f"Number of records: {info.total_records:,}\nSize on disk: {info.total_bytes / 1024**2:,.2f} MB"
)

Number of records: 1,000,000
Size on disk: 32.46 MB


We can see how many rows we're about to get, and the amount of data that will be
transferred over the network

In [6]:
import json

json.loads(info.app_metadata)

{'description': 'NYC CitiBike trips'}

The server can choose to send arbitrary metadata, which we can inspect here.
In this case, we get a nice description of the dataset.

## Discoverability

Discoverability is a key feature of Arrow Flight. For example, we can easily list all the datasets
we have available.

In [7]:
available_datasets = list(client.list_flights())
available_datasets

[<pyarrow.flight.FlightInfo schema=ride_id: large_string
   -- field metadata --
   PARQUET:field_id: '1'
 rideable_type: large_string
   -- field metadata --
   PARQUET:field_id: '2'
 started_at: timestamp[us]
   -- field metadata --
   PARQUET:field_id: '3'
 ended_at: timestamp[us]
   -- field metadata --
   PARQUET:field_id: '4'
 start_station_name: large_string
   -- field metadata --
   PARQUET:field_id: '5'
 start_station_id: large_string
   -- field metadata --
   PARQUET:field_id: '6'
 end_station_name: large_string
   -- field metadata --
   PARQUET:field_id: '7'
 end_station_id: large_string
   -- field metadata --
   PARQUET:field_id: '8'
 start_lat: double
   -- field metadata --
   PARQUET:field_id: '9'
 start_lng: double
   -- field metadata --
   PARQUET:field_id: '10'
 end_lat: double
   -- field metadata --
   PARQUET:field_id: '11'
 end_lng: double
   -- field metadata --
   PARQUET:field_id: '12'
 member_casual: large_string
   -- field metadata --
   PARQUET:field_i

### Listing with criteria

We can also apply send `criteria`, basically a set of bytes the server has implemented. In this case, the server has
implemented a matching search, so we can ask for all datasets that contain the string "rid"

In [8]:
list(client.list_flights(b"rid"))

[<pyarrow.flight.FlightInfo schema=ride_id: large_string
   -- field metadata --
   PARQUET:field_id: '1'
 rideable_type: large_string
   -- field metadata --
   PARQUET:field_id: '2'
 started_at: timestamp[us]
   -- field metadata --
   PARQUET:field_id: '3'
 ended_at: timestamp[us]
   -- field metadata --
   PARQUET:field_id: '4'
 start_station_name: large_string
   -- field metadata --
   PARQUET:field_id: '5'
 start_station_id: large_string
   -- field metadata --
   PARQUET:field_id: '6'
 end_station_name: large_string
   -- field metadata --
   PARQUET:field_id: '7'
 end_station_id: large_string
   -- field metadata --
   PARQUET:field_id: '8'
 start_lat: double
   -- field metadata --
   PARQUET:field_id: '9'
 start_lng: double
   -- field metadata --
   PARQUET:field_id: '10'
 end_lat: double
   -- field metadata --
   PARQUET:field_id: '11'
 end_lng: double
   -- field metadata --
   PARQUET:field_id: '12'
 member_casual: large_string
   -- field metadata --
   PARQUET:field_i

In [9]:
Server.list_flights??

Signature:
Server.list_flights(
    self,
    _: pyarrow._flight.ServerCallContext,
    criteria: bytes,
) -> collections.abc.Iterator[pyarrow._flight.FlightInfo]
Source:   
    @handle_flight_errors
    def list_flights(
        self, _: flight.ServerCallContext, criteria: bytes
    ) -> Iterator[flight.FlightInfo]:
        """Flight has native support for data discovery. The client can ask for all available
         flights and can send criteria to filter, where the criteria implementation is up to
        the implementer.
        """
        filter_match = criteria.decode("utf-8")
        log = logger.bind(filter_match=filter_match, method="list_flights")
        # catalog.list_tables returns a tuple of (namespace, table_name)
        table_identifiers = [
            ".".join(t)
            for t in self._catalog.list_tables(self._namespace)
            if filter_match.lower() in t[1].lower()
        ]
        log.info("listing flights")
        for identifier in table_identifiers:

In [10]:
Server._make_flight_info??

Signature: Server._make_flight_info(self, identifier: str) -> pyarrow._flight.FlightInfo
Source:   
    def _make_flight_info(self, identifier: str) -> flight.FlightInfo:
        """
        FlightInfo is the metadata for a dataset. We make it from the Iceberg table metadata
        """
        log = logger.bind(identifier=identifier, method="_make_flight_info")
        try:
            table = self._catalog.load_table(identifier)
        except NoSuchTableError:
            log.error("No such table")
            raise flight.FlightServerError(f"{identifier} not found")
        request = GetDatasetRequest(identifier=identifier)
        ticket = flight.Ticket(request.model_dump_json().encode("utf-8"))
        log.debug("ticket encoded", request=request)

        endpoints = [
            flight.FlightEndpoint(
                ticket=ticket,
                locations=[self._location, *self._workers],
            )
        ]

        snapshot = table.current_snapshot()
        num_rows = 

### FlightDescriptor - Asking for a specific dataset

To get a specific dataset, we can use a FlightDescriptor, basically a human-readable way of
communicating the dataset we want.

There are two types of FlightDescriptor, the `path` and the `command`.

Semantically, `path` relates to a location of data such as the name of a table or file.
`command` is a more implementation-specific approach, where the server can interpret the command
and execute it. This allows us to implement some powerful DSLs on top of Flight as needed.

Note that like REST and HTTP verbs (like GET, POST, PATCH), we, as implementors of a Flight server,
need to think about what the semantics are, but Arrow Flight doesn't impose any restrictions.

Let's go through the download process again, step-by-step.


## FlightInfo

First we need to get the `FlightInfo` about the dataset. We know that our data has the path
`message`, so we create a `FlightDescriptor` for that path

In [11]:
info = client.get_flight_info(flight.FlightDescriptor.for_path("rides"))

In [12]:
Server.get_flight_info??

Signature:
Server.get_flight_info(
    self,
    _: pyarrow._flight.ServerCallContext,
    descriptor: pyarrow._flight.FlightDescriptor,
) -> pyarrow._flight.FlightInfo
Source:   
    @handle_flight_errors
    def get_flight_info(
        self, _: flight.ServerCallContext, descriptor: flight.FlightDescriptor
    ) -> flight.FlightInfo:
        """The client can ask for the metadata for a dataset by calling get_flight_info.
        They will use a human-readable FlightDescriptor to describe the dataset they want.
        The FlightDescriptor can either be a path, or a command, but the definition is up to the
        implementer.

        The job of the FlightInfo is to return to the client where the data can be fetched
        from, build a Ticket for the Client to use to ask for the actual data, and provide
        some metadata such as the schema, number of rows, and size.
        """
        identifier = f"{self._namespace}.{descriptor.path[0].decode('utf-8')}"
        log = logger.b

We can also use the `FlightDescriptor` to get the schema for the datasetb

In [13]:
schema = client.get_schema(flight.FlightDescriptor.for_path("rides"))
schema

<pyarrow.flight.SchemaResult schema=(ride_id: large_string
  -- field metadata --
  PARQUET:field_id: '1'
rideable_type: large_string
  -- field metadata --
  PARQUET:field_id: '2'
started_at: timestamp[us]
  -- field metadata --
  PARQUET:field_id: '3'
ended_at: timestamp[us]
  -- field metadata --
  PARQUET:field_id: '4'
start_station_name: large_string
  -- field metadata --
  PARQUET:field_id: '5'
start_station_id: large_string
  -- field metadata --
  PARQUET:field_id: '6'
end_station_name: large_string
  -- field metadata --
  PARQUET:field_id: '7'
end_station_id: large_string
  -- field metadata --
  PARQUET:field_id: '8'
start_lat: double
  -- field metadata --
  PARQUET:field_id: '9'
start_lng: double
  -- field metadata --
  PARQUET:field_id: '10'
end_lat: double
  -- field metadata --
  PARQUET:field_id: '11'
end_lng: double
  -- field metadata --
  PARQUET:field_id: '12'
member_casual: large_string
  -- field metadata --
  PARQUET:field_id: '13')>

In [14]:
Server.get_schema??

Signature:
Server.get_schema(
    self,
    _: pyarrow._flight.ServerCallContext,
    descriptor: pyarrow._flight.FlightDescriptor,
) -> pyarrow._flight.SchemaResult
Source:   
    @handle_flight_errors
    def get_schema(
        self, _: flight.ServerCallContext, descriptor: flight.FlightDescriptor
    ) -> flight.SchemaResult:
        """Get the schema of the dataset."""
        table_name = descriptor.path[0].decode("utf-8")
        log = logger.bind(table_name=table_name, method="get_schema")
        try:
            log.info("getting schema")
            table = self._catalog.load_table(f"{self._namespace}.{table_name}")
        except NoSuchTableError:
            log.error("no such table")
            raise flight.FlightServerError(f"{table_name} not found")

        schema = table.schema().as_arrow()
        return flight.SchemaResult(schema)
File:      /app/.venv/lib/python3.13/site-packages/flight_server/server.py
Type:      function

## Tickets and Locations

One detail we have left out, is that `FlightInfo` also contains the endpoints our client should use
to actually get the data.

Arrow Flight works in a client/server model, a coordinator/worker model
or a mixture of both.

### Endpoints
Let's take a closer look at the `endpoint` again

In [15]:
info.endpoints

[<pyarrow.flight.FlightEndpoint ticket=<pyarrow.flight.Ticket ticket=b'{"identifier":"trips.rides","columns":["*"],"filters":null}'> locations=[<pyarrow.flight.Location b'grpc://0.0.0.0:7000'>] expiration_time=None app_metadata=b''>]

`FlightInfo` will contain a list of `endpoints` which the client can connect
to in order to get the data, and a corresponding `Ticket` and `Location` which the server has sent.

Each endpoint represents a part of the data - if the server sends multiple endpoints, that means we can create a client per endpoint
and download that data in parallel.

In [16]:
info.endpoints[0]

<pyarrow.flight.FlightEndpoint ticket=<pyarrow.flight.Ticket ticket=b'{"identifier":"trips.rides","columns":["*"],"filters":null}'> locations=[<pyarrow.flight.Location b'grpc://0.0.0.0:7000'>] expiration_time=None app_metadata=b''>

### The Ticket

The `Ticket` is metadata the server needs to fetch the correct data, and the
implementation of that metadata is up to the implementer, Arrow Flight does not prescribe anything. 

In this case, we've gone with simple JSON, but it
could be a Protobuf message, an Avro message or any other set of bytes.

The client doesn't need to care about the contents of the `Ticket`, the server is in charge of
providing the correct metadata. 

In [17]:
info.endpoints[0].ticket

<pyarrow.flight.Ticket ticket=b'{"identifier":"trips.rides","columns":["*"],"filters":null}'>

### Endpoint Location

The `Location` is where the `Ticket` can be used, and here the server can also provide multiple options.

It could be providing different geographical locations, so the client can pick the closest, or it could be simply a set of mirrors.

In [18]:
info.endpoints[0].locations

[<pyarrow.flight.Location b'grpc://0.0.0.0:7000'>]

In this case, we have a single endpoint, with a JSON-based ticket, so a simple client/server setup.

We need to grab the ticket from the endpoint, which we can use to get a reader, which we can use
to stream the data

## Do_get - fetch the data

To actually read the data, we call `do_get` with a `Ticket`, which returns a reader, representing a stream of data.
We can then choose to `read_chunk`s from the server or just `read_all`.

Here we call `read_chunk` to grab the next batch from the server.

In [21]:
reader = client.do_get(info.endpoints[0].ticket)

# Stream a chunk of data from the server
chunk = reader.read_chunk()

pl.from_arrow(chunk.data)

ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
str,str,datetime[μs],datetime[μs],str,str,str,str,f64,f64,f64,f64,str
"""85744AF35D7F2DF5""","""electric_bike""",2026-01-02 05:36:24.539,2026-01-02 05:42:21.153,"""W 42 St & 8 Ave""","""6602.05""","""E 58 St & Madison Ave""","""6839.04""",40.75757,-73.990985,40.763026,-73.972095,"""member"""
"""9D18958E5788880B""","""electric_bike""",2026-01-02 15:15:11.915,2026-01-02 15:18:40.462,"""Division St & Bowery""","""5311.08""","""Clinton St & Grand St""","""5303.06_""",40.71419,-73.99673,40.715738,-73.98699,"""member"""
"""B050891B7B009EE5""","""electric_bike""",2026-01-12 10:12:51.453,2026-01-12 10:16:55.731,"""Broadway & 31 St""","""6789.08""","""35 Ave & 37 St""","""6563.12""",40.76194,-73.92513,40.755733,-73.923661,"""member"""
"""0B6D7938C4EF1668""","""electric_bike""",2026-01-01 01:03:29.712,2026-01-01 01:05:37.341,"""34 St & 35 Ave""","""6605.08""","""35 St & Broadway""","""6750.16""",40.756933,-73.926223,40.760339,-73.922243,"""member"""
"""95415F60C7120CC3""","""electric_bike""",2026-01-03 19:55:51.636,2026-01-03 20:14:57.964,"""E 6 St & Ave B""","""5584.04""","""W 55 St & 6 Ave""","""6809.09""",40.724537,-73.981854,40.763189,-73.978434,"""member"""
…,…,…,…,…,…,…,…,…,…,…,…,…
"""92FFE092FE0E6139""","""electric_bike""",2026-01-02 16:24:25.343,2026-01-02 16:28:35.045,"""E 68 St & 3 Ave""","""6896.16""","""E 78 St & 2 Ave""","""7057.07""",40.767128,-73.962246,40.772797,-73.955778,"""member"""
"""68CD58DDAF4B04EB""","""classic_bike""",2026-01-02 15:37:47.637,2026-01-02 15:41:54.836,"""Tilden Ave & Lott St""","""3214.04""","""Beverley Rd & Nostrand Ave""","""3132.09""",40.64661,-73.95401,40.64507,-73.9488,"""member"""
"""11CBDE5B1F5A43B3""","""electric_bike""",2026-01-13 11:57:04.963,2026-01-13 12:01:21.820,"""N 7 St & Driggs Ave""","""5340.01""","""S 4 St & Wythe Ave""","""5204.05""",40.716967,-73.956388,40.712859,-73.965903,"""member"""


In [22]:
# We're not intending to read more data from the reader, so cancel the rest of the request
reader.cancel()

In [20]:
Server.do_get??

Signature:
Server.do_get(
    self,
    _: pyarrow._flight.ServerCallContext,
    ticket: pyarrow._flight.Ticket,
) -> pyarrow._flight.FlightDataStream
Source:   
    @handle_flight_errors
    def do_get(
        self, _: flight.ServerCallContext, ticket: flight.Ticket
    ) -> flight.FlightDataStream:
        """
        When a client calls get_flight_info, it will get a Ticket which we defined.
        The Ticket describes to the server how to get the data and contains an arbitrary payload
        only meant for the server to understand.
        """
        # We decided on JSON for the ticket payload, so we decode it here.
        request = GetDatasetRequest.model_validate_json(ticket.ticket.decode("utf-8"))
        log = logger.bind(request=request, method="do_get")
        table = self._catalog.load_table(request.identifier)

        filters = AlwaysTrue() if request.filters is None else request.filters

        reader = table.scan(
            selected_fields=request.columns,
    

## Summary of fetching data
![server_client](images/server_client.png)

## Do_Put - uploading data

We can also choose to have our Arrow Flight implement uploading data, using the same streaming mechanism as before, just going the other way

We have some campaign data to go along with our messages, so let's read in the CSV. Unsurprisingly,
Arrow Flight expects the data in Arrow format.

In [31]:
import pyarrow.csv as pc

schema = client.get_schema(flight.FlightDescriptor.for_path("rides"))

If I didn't pass the schema, the CSV inference would have gotten some things wrong - good thing Arrow Flight lets us communicate schemas!

In [35]:
rides = pc.read_csv("/app/data/202601-citibike-tripdata_2.csv", 
                    convert_options=pc.ConvertOptions(column_types=schema.schema))
rides

pyarrow.Table
ride_id: large_string
rideable_type: large_string
started_at: timestamp[us]
ended_at: timestamp[us]
start_station_name: large_string
start_station_id: large_string
end_station_name: large_string
end_station_id: large_string
start_lat: double
start_lng: double
end_lat: double
end_lng: double
member_casual: large_string
----
ride_id: [["0179695DC1F74E4E","5D0645202DDFA256","4572C883BF0165E4","3575D616D0179F57","895D2DC8F8BA4177",...,"3DCA3EDE80E9948A","57F14F28943FADA4","A028B11B6E6E310B","87EB1F35A4532C39","2E342B204BD598F4"],["10E677A6090F2FC8","7CCEBDFCB398B98F","A6B15E890C35A5BC","05C53F2DCB460724","C894198E220B739C",...,"F7C9AAE277D02D61","448DB87E03E2ECB1","EC97DB2EB09C8377","FA1433951D99DCD0","7B8D9BCDA7EBDEC0"],...,["9CAEA773CCC9D5E8","EB171534A17AB11C","3321A9330F939313","2094E1627F283FFF","BB13AE0BDAFB1513",...,"B2DA4E2EA80FB42E","206FA38A2FE59920","AD08EA3467A0A4D0","7ADB510C1A3550EB","8815947753DD777B"],["C624FC15D8DA228F","4ECA08A178313D16","1AE4822D91B6FEBC","

Next, we create a FlightDescriptor for the path we want to store the data in, as well as the schema
 of the data. Since Arrow Flight is a streaming the data, it needs to know the schema upfront.

Arrow Flight also uses the bidirectional streaming features of GRPC, so we have both a reader and
a writer as the return value

In [36]:
# Typehinting in Pyarrow library is still work in progress, best to help it along sometimes
writer: flight.FlightStreamWriter
reader: flight.FlightMetadataReader
writer, reader = client.do_put(
    flight.FlightDescriptor.for_path("rides"), rides.schema
)

In [37]:
# Since we're sending a GRPC stream, we need to tell the server that we're done writing before we can start reading
writer.write_table(rides)
writer.done_writing()

In [38]:
result = reader.read()
writer.close()

In [39]:
result.to_pybytes().decode("utf-8")

'{"total_rows":816391}'

In [40]:
Server.do_put??

Signature:
Server.do_put(
    self,
    _: pyarrow._flight.ServerCallContext,
    descriptor: pyarrow._flight.FlightDescriptor,
    reader: pyarrow._flight.FlightStreamReader,
    writer: pyarrow._flight.FlightMetadataWriter,
)
Source:   
    @handle_flight_errors
    def do_put(
        self,
        _: flight.ServerCallContext,
        descriptor: flight.FlightDescriptor,
        reader: flight.FlightStreamReader,
        writer: flight.FlightMetadataWriter,
    ):
        """Do_put is responsible for writing the data to storage. The client will send an
        Arrow Table, and the server will write it to storage. It can also send metadata back
        to the client, such as the number of rows written.
        """
        table_name = f"{self._namespace}.{descriptor.path[0].decode('utf-8')}"
        log = logger.bind(table_name=table_name, method="do_put")
        if not self._catalog.table_exists(table_name):
            log.info("creating table")
            try:
                if

Now that it has been created, we can check the `FlightInfo` to see that it has been correctly registered

In [29]:
info = client.get_flight_info(flight.FlightDescriptor.for_path("campaigns"))

In [30]:
info

<pyarrow.flight.FlightInfo schema=id: int64
campaign_type: string
channel: string
topic: string
started_at: timestamp[ns]
finished_at: timestamp[ms]
total_count: int64
ab_test: bool
warmup_mode: bool
hour_limit: int64
subject_length: double
subject_with_personalization: bool
subject_with_deadline: bool
subject_with_emoji: bool
subject_with_bonuses: bool
subject_with_discount: bool
subject_with_saleout: bool
is_test: bool
position: int64 descriptor=<pyarrow.flight.FlightDescriptor path=[b'campaigns']> endpoints=[<pyarrow.flight.FlightEndpoint ticket=<pyarrow.flight.Ticket ticket=b'{"name":"campaigns","bucket":"events","file_name":"campaigns.parquet","description":null,"file_type":"parquet","num_partitions":1,"num_rows":1907,"serialized_size":61888}'> locations=[<pyarrow.flight.Location b'grpc://0.0.0.0:7000'>] expiration_time=None app_metadata=b''>] total_records=1907 total_bytes=61888 ordered=False app_metadata=b'{"description": null}'>

And we can of course fetch it

In [31]:
reader: flight.FlightStreamReader = client.do_get(info.endpoints[0].ticket)

In [32]:
pl.from_arrow(reader.read_all())

id,campaign_type,channel,topic,started_at,finished_at,total_count,ab_test,warmup_mode,hour_limit,subject_length,subject_with_personalization,subject_with_deadline,subject_with_emoji,subject_with_bonuses,subject_with_discount,subject_with_saleout,is_test,position
i64,str,str,str,datetime[ns],datetime[ms],i64,bool,bool,i64,f64,bool,bool,bool,bool,bool,bool,bool,i64
63,"""bulk""","""mobile_push""","""sale out""",2021-04-30 07:22:36.615023,2021-04-30 07:23:41,48211,null,false,null,146.0,false,false,true,false,false,false,null,null
64,"""bulk""","""mobile_push""","""sale out""",2021-04-30 09:02:50.817227,2021-04-30 09:04:08,1037337,null,false,null,97.0,false,false,true,false,false,false,null,null
78,"""bulk""","""mobile_push""","""sale out""",2021-05-06 07:14:10.533318,2021-05-06 07:15:17,70080,null,false,null,146.0,false,false,true,false,false,false,null,null
79,"""bulk""","""mobile_push""","""sale out""",2021-05-06 09:03:56.486750,2021-05-06 09:42:15,921838,null,false,null,97.0,false,false,true,false,false,false,null,null
89,"""bulk""","""mobile_push""","""""",2021-05-07 11:54:06.168664,2021-05-07 11:54:38,45503,null,false,null,109.0,false,true,true,false,false,false,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
179,"""transactional""","""email""","""profile updated""",null,null,null,null,null,null,134.0,false,false,false,false,false,false,null,null
35,"""transactional""","""email""","""order reminder""",null,null,null,null,null,null,88.0,false,false,false,false,false,false,null,null
57,"""transactional""","""email""","""order reminder""",null,null,null,null,null,null,88.0,false,false,false,false,false,false,null,null


# More bonus stuff - Actions

Arrow Flight has a concept of registered `Actions` - think of them as arbitrary Commands the server
has implemented. We can use these to perform administrative tasks, such as updating a dataset description, or setting permissions.

We can have as many actions as we want, and the server can choose to implement them however it wants.

Flight includes native discoverability for actions, so let's use that to see what we can do.

In [41]:
import json
import pprint

for action in client.list_actions():
    print(f"Action Type: {action.type}")
    pprint.pprint(json.loads(action.description))
    print()

Action Type: create_namespace
{'description': 'Create a new namespace',
 'schema': {'properties': {'name': {'title': 'Name', 'type': 'string'}},
            'required': ['name'],
            'title': 'CreateNamespaceRequest',
            'type': 'object'}}

Action Type: update_description
{'description': 'Update a dataset description',
 'schema': {'properties': {'description': {'title': 'Description',
                                           'type': 'string'},
                           'name': {'title': 'Name', 'type': 'string'}},
            'required': ['name', 'description'],
            'title': 'UpdateDatasetRequest',
            'type': 'object'}}

Action Type: delete_dataset
{'description': 'Delete a dataset',
 'schema': {'properties': {'name': {'title': 'Name', 'type': 'string'}},
            'required': ['name'],
            'title': 'DeleteDatasetRequest',
            'type': 'object'}}

Action Type: get_active_user
{'description': 'Get the active user', 'schema': None}



In [34]:
Server.list_actions??

Signature: Server.list_actions(self, context: pyarrow._flight.ServerCallContext) -> Iterator[pyarrow._flight.ActionType]
Source:   
    def list_actions(
        self, context: flight.ServerCallContext
    ) -> Iterator[flight.ActionType]:
        """Flight has native support for actions.
        Actions are a way to perform arbitrary operations, and list_actions is a discovery
        mechanism for the client to find out what actions are available.
        """
        actions = [
            (
                "update_description",
                json.dumps(
                    {
                        "description": "Update a dataset description",
                        "schema": UpdateDatasetRequest.model_json_schema(),
                    }
                ),
            ),
            (
                "delete_dataset",
                json.dumps(
                    {
                        "description": "Delete a dataset",
                        "schema": DeleteDatasetReque

The server is advertising an `update_description` action along with a description of the action.

To demonstrate, let's fix the description, since it's not currently very good
An Action consists of an `action_type` and a body of bytes, which are then parsed by the server.

In our case, we have our UpdateDescription Pydantic model contract shared between Client and Server that is expected from the server

In [43]:
from flight_server.models import UpdateDatasetRequest

dataset = UpdateDatasetRequest(
    name="rides",
    description="This dataset covers the NYC CitiBike trip rides",
)

In [46]:
action = flight.Action(
    action_type="update_description", buf=dataset.model_dump_json().encode("utf-8")
)
result = client.do_action(action)

The return value is a generator of `flight.Result`, which is a wrapper around some bytes, so we
need to serialize it to a string to see the result.

In [47]:
next(result).body.to_pybytes().decode("utf-8")

'Updated dataset rides description'

In [49]:
client.get_flight_info(flight.FlightDescriptor.for_path("rides")).app_metadata

b'{"description": "This dataset covers the NYC CitiBike trip rides"}'

In [39]:
Server.do_action??

Signature:
Server.do_action(
    self,
    context: pyarrow._flight.ServerCallContext,
    action: pyarrow._flight.Action,
) -> Iterator[bytes]
Source:   
    def do_action(
        self, context: flight.ServerCallContext, action: flight.Action
    ) -> Iterator[bytes]:
        """When the client wants to perform an action, it will send an Action via do_action.
        What that action does is completely up to the implementation.
        """
        match action.type:
            case "update_description":
                request = UpdateDatasetRequest.model_validate_json(
                    action.body.to_pybytes().decode("utf-8")
                )
                with self._dataset_repo as repo:
                    repo.update_dataset(request)
                yield f"Updated dataset {request.name} description".encode("utf-8")
            case "delete_dataset":
                request = DeleteDatasetRequest.model_validate_json(
                    action.body.to_pybytes().decode("utf

# Do_exchange - Request/Response in the same call

The `do_put` and `do_get` are one-way - data is either coming from the server or being sent to the server. 

`do_exchange` is when we have some data we want the server to process and return to us.

In this example, we can send some ride data to the server to calculate a metric, and return that to us. 

As an example, we select a subset of the data, and then we ask the server to do the calculation for us.

In [50]:
info = client.get_flight_info(flight.FlightDescriptor.for_path("rides"))
table = client.do_get(info.endpoints[0].ticket).read_all()
df = pl.from_arrow(table)

To simulate getting a new dataset, we select out the "bulk" message type

In [65]:
classic_bikes = df.filter(pl.col("rideable_type") == "electric_bike").to_arrow()

In [71]:
classic_bikes

pyarrow.Table
ride_id: large_string
rideable_type: large_string
started_at: timestamp[us]
ended_at: timestamp[us]
start_station_name: large_string
start_station_id: large_string
end_station_name: large_string
end_station_id: large_string
start_lat: double
start_lng: double
end_lat: double
end_lng: double
member_casual: large_string
----
ride_id: [["5D0645202DDFA256","3575D616D0179F57","895D2DC8F8BA4177","C812035F0F77F0FE","9818B3A1B4530667",...,"E771EB98769C54D0","5E1BC2953B476C78","CC254688E4151B73","63160D4898957580","B6E9B9CD81F0F21B"]]
rideable_type: [["electric_bike","electric_bike","electric_bike","electric_bike","electric_bike",...,"electric_bike","electric_bike","electric_bike","electric_bike","electric_bike"]]
started_at: [[2026-01-09 10:14:39.226000,2026-01-06 05:57:18.257000,2026-01-02 16:19:07.373000,2026-01-11 12:31:20.262000,2026-01-06 14:56:24.245000,...,2026-01-02 12:42:51.450000,2026-01-06 15:55:23.634000,2026-01-05 22:21:08.536000,2026-01-10 11:38:56.010000,2026-01-06

Now we're ready to do our exchange.

In [72]:
# Do_exchange returns a writer and reader, similar to do_put.
# For the FlightDescriptor, we're demonstrating a command, which must be serialized to bytes
command = json.dumps({"metric": "manhattan_distance"}).encode("utf-8")
writer, reader = client.do_exchange(flight.FlightDescriptor.for_command(command))

# Since this is a stream, we need to describe the schema of what's to come.
writer.begin(classic_bikes.schema)
# Send the data
writer.write_table(classic_bikes)

# Tell the server we're done writing - ready for the response
writer.done_writing()

# Read the response
result = reader.read_all()

In [73]:
pl.from_arrow(result)

ride_id,manhattan_distance_km
str,f64
"""5D0645202DDFA256""",1.383785
"""3575D616D0179F57""",4.074762
"""895D2DC8F8BA4177""",1.448487
"""C812035F0F77F0FE""",5.40201
"""9818B3A1B4530667""",0.991532
…,…
"""E771EB98769C54D0""",1.124387
"""5E1BC2953B476C78""",0.993267
"""CC254688E4151B73""",0.930239


In [44]:
Server.do_exchange??

Signature:
Server.do_exchange(
    self,
    context: pyarrow._flight.ServerCallContext,
    descriptor: pyarrow._flight.FlightDescriptor,
    reader: pyarrow._flight.FlightStreamReader,
    writer: pyarrow._flight.FlightStreamWriter,
)
Source:   
    def do_exchange(
        self,
        context: flight.ServerCallContext,
        descriptor: flight.FlightDescriptor,
        reader: flight.FlightStreamReader,
        writer: flight.FlightStreamWriter,
    ):
        """Flight can receive and send data within the same call using do_exchange.
        This is commonly used for enriching data, such as calling an ML model to enrich the data,
        or calculate some metric on the data.
        """
        cmd: str = descriptor.command.decode("utf-8")
        match cmd:
            case "ctr":
                sample_ctr_data: pa.Table = reader.read_all()
                df = pl.DataFrame(sample_ctr_data)
                min_date = cast(dt.date, df["date"].min())
                max_date = 

# Conclusion

Apache Arrow is a powerful protocol to have in our back pockets. Whenever you see the need for a client/server implementation, that requires working with large amounts of data, or that would benefit from a RPC-style implementation, reach for Arrow Flight.

It is multi-lingual, supports a distributed architecture natively and allows you to be end-to-end Arrow. It will always be better than ODBC/JSON serialization for large datasets!

In this short demo, we have looked at the basic mechanisms of using Arrow Flight, but these building blocks are something you can use to scale to almost any usecase

